In [ ]:
import time
import gc
import os
import codecs
import re
import base64
import win32com.client
import pandas as pd
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

# ===== إعدادات =====
SENDER_EMAIL = 'xxxxx'
RESTART_INTERVAL = 50
EMAILS_PER_HOUR = 150
emails_sent_this_hour = 0

today = datetime.now().strftime("%d-%m-%Y")

# ============================================================
# ===== 1) تحميل البيانات =====
# ============================================================
print("="*60)
print("📌 STEP 1: Loading data...")
print("="*60)

data = pd.read_excel(r"C:\Users\int34\Desktop\pending_invoives_Vat.xlsx")
print(f"✅ Loaded pending_invoives.xlsx: {len(data)} rows")

data = data.drop_duplicates(subset=['BookingCode'])
data['Supplier Name'] = data['Supplier Name'].str.replace('/', '-')
data['Lower Supplier Name'] = data['Supplier Name'].apply(lambda x: x.lower().strip().replace("  ", " "))

emails = pd.read_excel(r"W:\Invoices KSA VAT\All KSA suppliers Mails.xlsx")
print(f"✅ Loaded All KSA suppliers Mails.xlsx: {len(emails)} rows")

emails['Lower Supplier Name'] = emails['Supplier Name'].apply(lambda x: x.lower().strip().replace("  ", " "))

# ✅ دمج Email + CC
data = data.merge(emails[['Lower Supplier Name', 'Email', 'CC']], on='Lower Supplier Name', how='left')

# ✅ تنظيف Email + CC
data['Email'] = data['Email'].fillna('').astype(str).str.strip()
data['CC'] = data['CC'].fillna('').astype(str).str.strip()
data['Email'] = data['Email'].replace(['nan', 'None', 'NaN'], '')

data.drop(['Lower Supplier Name'], axis=1, inplace=True)
data = data.reset_index(drop=True)

# ✅ الفلترة
send_df = data[data['Email'] != ''].copy()
NoEmails_df = data[data['Email'] == ''].copy()

print(f"📧 Suppliers with emails: {send_df['Supplier Name'].nunique()}")
print(f"❌ Suppliers without emails: {NoEmails_df['Supplier Name'].nunique()}")

# ✅✅✅ حفظ الملفات - السطور الناقصة
send_df.to_excel(f"C:\\Users\\int34\\Downloads\\SendPendingInvoices{today}.xlsx", index=False)
NoEmails_df.to_excel(f"C:\\Users\\int34\\Downloads\\NotSentPendingInvoices{today}.xlsx", index=False)

print(f"💾 Saved: SendPendingInvoices{today}.xlsx")
print(f"💾 Saved: NotSentPendingInvoices{today}.xlsx")

# ============================================================
# ✅✅✅ السطر الناقص - استخرج أسماء الـ Suppliers
# ============================================================
send_df['Supplier Name'] = send_df['Supplier Name'].apply(lambda x: x.title())
hotels = send_df['Supplier Name'].unique().tolist()
hotels_len = len(hotels)
print(f"✅ Total suppliers to process: {hotels_len}")

# ============================================================
# ===== 2) بناء signature =====
# ============================================================
print("\n" + "="*60)
print("📌 STEP 2: Building signature...")
print("="*60)

SIGNATURE_NAME = 'Invoices_ETG (Invocies@etg.sa)'

def img_to_b64(img_path):
    ext = os.path.splitext(img_path)[1].lower().strip('.')
    if ext == 'jpg':
        ext = 'jpeg'
    elif ext == 'svg':
        ext = 'svg+xml'
    with open(img_path, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode('utf-8')
    return f'data:image/{ext};base64,{b64}'

try:
    user_profile = os.environ['USERPROFILE']
    signatures_dir = os.path.join(user_profile, 'AppData', 'Roaming', 'Microsoft', 'Signatures')
    signature_files_dir = os.path.join(signatures_dir, f'{SIGNATURE_NAME}_files')
    
    logo_big = img_to_b64(os.path.join(signature_files_dir, 'image001.png'))
    orange_bar = img_to_b64(os.path.join(signature_files_dir, 'image002.png'))
    partners = img_to_b64(os.path.join(signature_files_dir, 'image003.jpg'))
    print("   ✅ Images loaded (logo + bar + partners)")
    
    signature_code = f"""
<table cellpadding="0" cellspacing="0" border="0" style="font-family: Arial, sans-serif; font-size: 12px; color: #333333; border-collapse: collapse;">
    <tr>
        <td style="vertical-align: middle; padding: 0 12px 0 0;">
            <img src="{logo_big}" width="196" height="91" alt="ET Global" style="display: block; border: 0;">
        </td>
        <td style="vertical-align: top; padding: 0 12px 0 0;">
            <img src="{orange_bar}" width="6" height="157" alt="" style="display: block; border: 0;">
        </td>
        <td style="vertical-align: top; padding: 0; line-height: 1.5;">
            <div style="color: #E87722; font-size: 18px; font-weight: bold; line-height: 1.1;">Payable Team</div>
            <div style="color: #333333; font-size: 12px; font-style: italic; margin-bottom: 3px;">Accountant</div>
            <div style="color: #1F4E79; font-size: 13px; font-weight: bold; margin-bottom: 8px;">ET Global</div>
            <table cellpadding="0" cellspacing="0" border="0" style="font-family: Arial, sans-serif; font-size: 11px; color: #333333; line-height: 1.7;">
                <tr>
                    <td style="padding: 1px 6px 1px 0; vertical-align: top; color: #1F4E79; font-size: 14px;">&#128241;</td>
                    <td style="padding: 1px 0; vertical-align: top;">Mob: +966543491855</td>
                </tr>
                <tr>
                    <td style="padding: 1px 6px 1px 0; vertical-align: top; color: #1F4E79; font-size: 14px;">&#9993;</td>
                    <td style="padding: 1px 0; vertical-align: top;"><a href="mailto:Invocies@etg.sa" style="color: #333333; text-decoration: none;">Invocies@etg.sa</a></td>
                </tr>
                <tr>
                    <td style="padding: 1px 6px 1px 0; vertical-align: top; color: #C00000; font-size: 14px;">&#128205;</td>
                    <td style="padding: 1px 0; vertical-align: top;">Office 218 Al-Baghdadiah Plaza Building, Al Madina Road - Al-Baghadadiah Al-Sharqya District - P.O. No 2344 Zip Code 23216 - Jeddah - Saudi Arabia.</td>
                </tr>
            </table>
        </td>
    </tr>
</table>
<div style="margin-top: 10px;">
    <img src="{partners}" width="616" height="77" alt="Partners" style="display: block; border: 0;">
</div>
"""
    print(f"✅ Signature built: {SIGNATURE_NAME}")
    print(f"📏 Signature size: {len(signature_code):,} chars")
    
except Exception as e:
    signature_code = "<br><br>Best Regards,<br>ET Global KSA"
    print(f"⚠️ Fallback signature: {e}")

# ============================================================
# ===== 3) دوال Outlook =====
# ============================================================
def wait_for_outlook(timeout=60):
    print("⏳ Waiting for Outlook to be ready...")
    start_time = time.time()
    while time.time() - start_time < timeout:
        try:
            outlook = win32com.client.Dispatch("Outlook.Application")
            accounts = outlook.Session.Accounts
            print("✅ Outlook is ready!")
            return outlook
        except:
            time.sleep(5)
    print("❌ Timeout")
    return None

def open_outlook():
    print("📌 Opening Outlook...")
    try:
        os.system("taskkill /f /im outlook.exe")
        time.sleep(5)
    except:
        pass
    try:
        os.startfile('outlook')
        print("   ✅ Started Outlook")
    except:
        os.system("start outlook")
        print("   ✅ Started Outlook with system command")
    time.sleep(10)
    return wait_for_outlook(timeout=90)

def find_sender_account(accounts):
    for acc in range(len(accounts)):
        if accounts[acc].DisplayName.lower() == SENDER_EMAIL.lower():
            return accounts[acc]
    return None

def restart_outlook():
    print("\n🔄 Restarting Outlook...")
    try:
        outlook = win32com.client.Dispatch("Outlook.Application")
        outlook.Quit()
        time.sleep(3)
    except:
        pass
    os.system("taskkill /f /im outlook.exe")
    time.sleep(5)
    gc.collect()
    try:
        os.startfile('outlook')
        print("   ✅ Started Outlook")
    except:
        os.system("start outlook")
        print("   ✅ Started Outlook with system command")
    time.sleep(10)
    outlook = wait_for_outlook(timeout=90)
    if outlook is None:
        return None
    
    accounts = outlook.Session.Accounts
    send_account_obj = find_sender_account(accounts)
    
    if send_account_obj is None:
        print(f"❌ '{SENDER_EMAIL}' NOT found after restart!")
        return None
    
    print(f"📧 Account: {send_account_obj.DisplayName}")
    return outlook, accounts, send_account_obj

# ============================================================
# ===== 4) فتح Outlook أول مرة =====
# ============================================================
outlook = open_outlook()
if outlook is None:
    print("❌ Cannot start. Exiting...")
    exit()

accounts = outlook.Session.Accounts

print("\n📋 Available accounts:")
for acc in range(len(accounts)):
    print(f"   [{acc}] {accounts[acc].DisplayName}")

send_account_obj = find_sender_account(accounts)

if send_account_obj is None:
    print(f"\n❌ '{SENDER_EMAIL}' NOT found in Outlook accounts!")
    exit()

print(f"\n✅ Will send from: {send_account_obj.DisplayName}")
print(f"📊 Total suppliers to process: {len(hotels)}")
print(f"🔄 Restart Outlook every {RESTART_INTERVAL} suppliers")
print(f"⏰ Wait 1 hour every {EMAILS_PER_HOUR} suppliers")

# ============================================================
# ===== 5) حلقة الإرسال =====
# ============================================================
for idx, i in enumerate(hotels):
    if emails_sent_this_hour >= EMAILS_PER_HOUR:
        print(f"\n{'='*60}")
        print(f"⏰ {EMAILS_PER_HOUR} suppliers sent this hour! Waiting 1 hour...")
        print(f"{'='*60}\n")
        
        for remaining in range(1800, 0, -1):   # sleep 30 min between 150 emails
            if remaining % 300 == 0:
                print(f"   ⏱️ {remaining//60} minutes remaining...")
            time.sleep(1)
        
        emails_sent_this_hour = 0
        print(f"✅ Hourly counter reset\n")
    
    print(f"\n📨 Email #{idx+1}: {i}")
    
    df = send_df[send_df["Supplier Name"] == i]
    if df.empty:
        print(f"⚠️ No data for: {i}")
        continue
    
    table = """
    <table border='1' style='border-collapse: collapse; width: 100%; font-family: Arial; font-size: 12px;'>
        <tr style='background-color: #04AA6D; color: white;'>
            <th style='border: 1px solid black; padding: 8px; text-align: center;'>BookingCode</th>
            <th style='border: 1px solid black; padding: 8px; text-align: center;'>First Name</th>
            <th style='border: 1px solid black; padding: 8px; text-align: center;'>Last Name</th>
            <th style='border: 1px solid black; padding: 8px; text-align: center;'>BeginTravelDate</th>
            <th style='border: 1px solid black; padding: 8px; text-align: center;'>EndTravelDate</th>
            <th style='border: 1px solid black; padding: 8px; text-align: center;'>Rate</th>
            <th style='border: 1px solid black; padding: 8px; text-align: center;'>Currency</th>
            <th style='border: 1px solid black; padding: 8px; text-align: center;'>Remarks</th>
        </tr>"""
    
    for k, v in df.iterrows():
        table += f"""
        <tr style='background-color: {'#f9f9f9' if k % 2 == 0 else 'white'};'>
            <td style='border: 1px solid black; padding: 8px; text-align: center;'>{df.loc[k]['BookingCode']}</td>
            <td style='border: 1px solid black; padding: 8px; text-align: center;'>{df.loc[k]['First Name']}</td>
            <td style='border: 1px solid black; padding: 8px; text-align: center;'>{df.loc[k]['Last Name']}</td>
            <td style='border: 1px solid black; padding: 8px; text-align: center;'>{df.loc[k]['BeginTravelDate']}</td>
            <td style='border: 1px solid black; padding: 8px; text-align: center;'>{df.loc[k]['EndTravelDate']}</td>
            <td style='border: 1px solid black; padding: 8px; text-align: center;'>{df.loc[k]['Net Cost']}</td>
            <td style='border: 1px solid black; padding: 8px; text-align: center;'>{df.loc[k]['Cost Currency']}</td>
            <td style='border: 1px solid black; padding: 8px; text-align: center;'>{df.loc[k]['Remarks']}</td>
        </tr>"""
    
    table += "</table>"
    
    try:
        hotel_mail = df["Email"].values[0]
        hotel_cc = df["CC"].values[0] if "CC" in df.columns else ""
        print(f"📧 Sending to: {hotel_mail}")
        print(f"📧 CC: {hotel_cc}")
        
        newMail = outlook.CreateItem(0)
        newMail._oleobj_.Invoke(*(64209, 0, 8, 0, send_account_obj))
        
        newMail.To = hotel_mail#'ahmed.saker@eetglobal.net'
        newMail.CC = str(hotel_cc) if hotel_cc else "" #'ahmed.saker@eetglobal.net'
        newMail.Subject = f"Pending Invoices - {i} - {today}"
        newMail.BodyFormat = 2
        
        try:
            newMail.Attachments.Add(r"Y:\Amr Atef\VCC Payment Autosend\uploads\resources\ET Global - CR.pdf")
            newMail.Attachments.Add(r"Y:\Amr Atef\VCC Payment Autosend\uploads\resources\ET Vat cetifcate.pdf")
        except Exception as e:
            print(f"⚠️ Attachments error: {e}")
        
        body = f"""
        <html>
        <body style='font-family: Arial;'>
        <p>Dear Partner,</p>
        <span>Greetings from ET Global KSA,</span>
        <p>I am writing to follow up on our previous requests for the outstanding invoices against the following reservations.</p>
        <p>We kindly request that you send the invoices as soon as possible, as we need them for our financial records. Your prompt attention to this matter would be greatly appreciated.</p>
        <p style='color:red'>Please make sure the invoices should be directed to ET Global KSA, ensuring that the name of our company is clearly stated and under our VAT Number</p>
        <p>We have provided our VAT Registration and Trade License for your reference.</p>
        
        {table}
        
        <p>Please let me know if you need any further details or information from my side to process the invoices.</p>
        <p>Best Regards</p>
        {signature_code}
        </body>
        </html>
        """
        
        newMail.HTMLBody = body
        #newMail.display()
        time.sleep(1)
        newMail.Send()
        print(f"✅ Email #{idx+1} sent!")
        
        emails_sent_this_hour += 1
        time.sleep(2)
        
        if (idx + 1) % RESTART_INTERVAL == 0:
            print(f"\n{'='*60}")
            print(f"🔄 {RESTART_INTERVAL} suppliers sent! Restarting Outlook...")
            print(f"{'='*60}\n")
            
            result = restart_outlook()
            
            if result is None:
                print("❌ Failed to restart. Trying emergency...")
                os.system("start outlook")
                time.sleep(30)
                outlook = wait_for_outlook(timeout=60)
                if outlook:
                    accounts = outlook.Session.Accounts
                    send_account_obj = find_sender_account(accounts)
                    if send_account_obj is None:
                        print("❌ Cannot find sender account. Stopping...")
                        break
                else:
                    print("❌ Cannot recover. Stopping...")
                    break
            else:
                outlook, accounts, send_account_obj = result
            
    except Exception as e:
        print(f"❌ Error with {i}: {e}")
        continue

print("\n" + "="*60)
print("✅ ALL DONE!")
print(f"📊 Total suppliers processed: {len(hotels)}")
print("="*60)

📌 STEP 1: Loading data...
ERROR! Session/line number was not unique in database. History logging moved to new session 453
✅ Loaded pending_invoives.xlsx: 6604 rows
✅ Loaded All KSA suppliers Mails.xlsx: 439 rows
📧 Suppliers with emails: 278
❌ Suppliers without emails: 6
💾 Saved: SendPendingInvoices20-10-2026.xlsx
💾 Saved: NotSentPendingInvoices20-10-2026.xlsx
✅ Total suppliers to process: 278

📌 STEP 2: Building signature...
   ✅ Images loaded (logo + bar + partners)
✅ Signature built: Invoices_ETG (Invocies@etg.sa)
📏 Signature size: 39,350 chars
📌 Opening Outlook...
   ✅ Started Outlook
⏳ Waiting for Outlook to be ready...
✅ Outlook is ready!

📋 Available accounts:
   [0] ahmed.saker@eetglobal.net
   [1] vcc@eetglobal.net
   [2] internetdept@eetglobal.net
   [3] Payable@etg.sa
   [4] Invocies@etg.sa

✅ Will send from: Invocies@etg.sa
📊 Total suppliers to process: 278
🔄 Restart Outlook every 50 suppliers
⏰ Wait 1 hour every 150 suppliers

📨 Email #1: H2H - Dotw (Sa)
📧 Sending to: mea.a